# colab-whisper-stt: Whisper for Korean/English mixed recordings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ehdtn0843ds-dss/colab-whisper-stt/blob/main/notebook/colab_whisper_stt.ipynb)

Transcribes recordings where Korean and English are mixed, such as a Korean lecture full of English terms or a meeting that switches to English partway through. It runs faster-whisper large-v3 on a free Colab T4, picks the language for each speech chunk, uses a hotword list for names and jargon, and saves progress in blocks so a dropped session resumes where it stopped.

Guide (English / 한국어): https://ehdtn0843ds-dss.github.io/colab-whisper-stt/

How to run:
1. `Runtime → Change runtime type → T4 GPU`
2. Run SETUP, then BOOTSTRAP. It creates `MyDrive/whisper_stt/` and copies the code and templates.
3. Put recordings in `MyDrive/whisper_stt/audio/<id>.m4a` and describe them in `code/sessions.json`.
4. Run MOUNT_AND_LOAD and RUN. A zip with the transcripts downloads when it finishes.

## 한·영 혼용 녹음용 Whisper

영어 용어가 계속 나오는 한국어 강의나 중간에 영어로 넘어가는 회의처럼 두 언어가 섞인 녹음을 전사한다. 무료 Colab T4에서 faster-whisper large-v3를 돌리고, 언어는 발화 조각마다 고른다. 이름과 전문 용어는 hotword 목록으로 넣고, 진행 상황은 블록 단위로 저장해서 연결이 끊겨도 이어서 돈다.

실행 순서:
1. 런타임 유형을 T4 GPU로 바꾼다.
2. SETUP, BOOTSTRAP 순서로 실행한다. `MyDrive/whisper_stt/`가 생기고 코드와 템플릿이 복사된다.
3. 녹음을 `MyDrive/whisper_stt/audio/<id>.m4a`에 넣고 `code/sessions.json`에 설명을 적는다.
4. MOUNT_AND_LOAD, RUN을 실행한다. 끝나면 전사 결과 zip이 내려받아진다.


In [ ]:
# SETUP
%pip -q install faster-whisper==1.2.1 av==16.0.1 ctranslate2==4.8.5
!nvidia-smi
import ctranslate2
import subprocess
import importlib.metadata
print(ctranslate2.get_cuda_device_count())
setup_info = {"cuda_device_count": ctranslate2.get_cuda_device_count(), "versions": {name: importlib.metadata.version(name) for name in ("faster-whisper", "ctranslate2", "av")}}
try:
    setup_info["nvidia_smi"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
except OSError as exc:
    setup_info["nvidia_smi_error"] = str(exc)
# CUDA 12 runtime libraries for ctranslate2
%pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12
import os, glob
_cuda_libs = sorted(glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/*/lib'))
os.environ['LD_LIBRARY_PATH'] = ':'.join(_cuda_libs + [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p])
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])
setup_info['ld_library_path'] = os.environ['LD_LIBRARY_PATH']


In [ ]:
# BOOTSTRAP (optional, run once): copy the pipeline code and templates from GitHub into Drive.
# Existing files in Drive are never overwritten, so your edited sessions.json/control files are safe.
import os, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/whisper_stt"
REPO = "/content/colab-whisper-stt"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ehdtn0843ds-dss/colab-whisper-stt", REPO], check=True)
os.makedirs(BASE + "/audio", exist_ok=True)
os.makedirs(BASE + "/code", exist_ok=True)
def copy_if_missing(src, dst):
    if os.path.exists(dst):
        print("keep  ", dst)
    else:
        shutil.copy2(src, dst)
        print("copied", dst)
for name in ("stt_core.py", "runner.py", "sessions.json"):
    copy_if_missing(f"{REPO}/code/{name}", f"{BASE}/code/{name}")
if not any(n.startswith("control_") and n.endswith(".json") for n in os.listdir(BASE)):
    copy_if_missing(f"{REPO}/examples/control_001_pilot.json", f"{BASE}/control_001_pilot.json")
print("Audio files found:", sorted(os.listdir(BASE + "/audio")))


In [ ]:
# MOUNT_AND_LOAD
import glob
import hashlib
import json
import os
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/whisper_stt"
_controls = sorted(glob.glob(BASE + "/control_*.json"))
if not _controls:
    raise FileNotFoundError("No control_*.json found in " + BASE)
_control_path = _controls[-1]
with open(_control_path, encoding="utf-8") as _stream:
    control = json.load(_stream)
control["_name"] = os.path.splitext(os.path.basename(_control_path))[0]
CODE = BASE + "/" + control["code_dir"]
print("Control:", os.path.basename(_control_path))
for _name in ("stt_core.py", "runner.py"):
    with open(CODE + "/" + _name, "rb") as _stream:
        print(_name, "SHA-256:", hashlib.sha256(_stream.read()).hexdigest())
if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No CUDA GPU available. Select a GPU runtime before running transcription.")
exec(compile(open(CODE + "/runner.py", encoding="utf-8").read(), "runner.py", "exec"))


In [ ]:
# RUN
main(BASE, control, CODE, setup_info)
